# Phase 1: Voice deepfake detection (AASIST3 on ASVspoof5)

Thin Colab driver. All real logic lives in `src/audio/` in the repo so it's
version-controlled and editable in VS Code / Claude Code instead of being
trapped in notebook cells. Set `REPO_URL` below, then run cells top to bottom.

If this is a **private** repo, put a GitHub personal access token in Colab's
Secrets panel (key icon in the left sidebar) named `GITHUB_TOKEN` before running.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "github.com/Ayat1807/ElderTrustShield.git"
REPO_DIR = Path('/content/repo')

if not REPO_DIR.exists():
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        clone_url = f"https://{token}@{REPO_URL}"
    except Exception:
        clone_url = f"https://{REPO_URL}"  # public repo, no token needed
    !git clone {clone_url} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.insert(0, str(REPO_DIR / 'src'))

In [ ]:
import torch
from audio import download, extract, labels, model as model_mod, inference, evaluate

PROTOCOL_BASE = Path('/content/asvspoof5')
DRIVE_BASE = Path('/content/drive/MyDrive/ElderTrustShield/data/audio/asvspoof5')
LOCAL_EXTRACT_DIR = Path('/content/flac_D_aa')
LOCAL_TAR = Path('/content/flac_D_aa.tar')
AASIST3_REPO_DIR = Path('/content/AASIST3')
CHECKPOINT_PATH = DRIVE_BASE / 'inference_checkpoint.pkl'
EXPECTED_AUDIO_COUNT = 47400

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE=', device)
if device.type != 'cuda':
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then rerun.')

In [ ]:
protocol_tar = download.download_protocols(PROTOCOL_BASE)
protocol_path = PROTOCOL_BASE / 'ASVspoof5.dev.track_1.tsv'
if not protocol_path.exists():
    import tarfile
    with tarfile.open(protocol_tar, 'r') as tf:
        tf.extractall(PROTOCOL_BASE)

drive_archive = download.download_audio_shard(DRIVE_BASE)
extract.extract_shard(drive_archive, LOCAL_TAR, LOCAL_EXTRACT_DIR, EXPECTED_AUDIO_COUNT)

labels_csv = DRIVE_BASE / 'flac_D_aa_labels.csv'
labels.build_labels_csv(protocol_path, LOCAL_EXTRACT_DIR, labels_csv)

In [ ]:
aasist3 = model_mod.load_model(AASIST3_REPO_DIR, device)
true_labels, predicted_labels, bonafide_scores, attack_algorithms = inference.run_inference(
    aasist3, device, labels_csv, LOCAL_EXTRACT_DIR, CHECKPOINT_PATH
)

In [ ]:
metrics = evaluate.compute_metrics(true_labels, predicted_labels, bonafide_scores, attack_algorithms)
evaluate.print_report(metrics)